# 5-Fold Grouped Cross-Validation

Robustness check for *Comparative Analysis of CNN and Vision Transformer Architectures
for Explainable Endometriosis Classification in Laparoscopic Images*.

The main results use one held-out, case-wise test split. A reviewer will ask whether
those numbers are a lucky split, so this notebook retrains **ResNet50, EfficientNet-B0
and ViT-B/16** across **5 grouped folds** over all 746 images and reports mean ± std.

**Rules that must not be broken**
- `StratifiedGroupKFold` over the 129 case/video groups — never plain `KFold`.
  A frame from one surgery must never appear in two folds, or the result is leakage.
- Each fold carves its **own validation set, also group-wise**, for early stopping.
  The fold's test part is never used for model selection.
- `seed = 42`, identical hyperparameters in every fold.
- **Per-image predictions are saved**, not just averages — significance tests need them.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then upload
`glenda_balanced.zip` to the session (left sidebar → Files → Upload).

Expect roughly **1.5–2 hours** for all 15 runs (3 models × 5 folds). Results are written
after every single run, so a disconnect loses at most one run — just re-run the training
cell and it resumes.

In [ ]:
!unzip -q -o glenda_balanced.zip -d data
!ls data

## 1. Setup

If Drive mounts, results are written there so a Colab disconnect cannot lose them.

In [ ]:
import os, random, time, csv
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
N_FOLDS = 5
IMG, BATCH = 224, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
MAX_EPOCHS, PATIENCE = 30, 6

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if device.type != "cuda":
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, or this will take many hours.")

# persist results to Drive when available, so a disconnect does not lose finished folds
OUT_DIR = Path("/content")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/endometriosis_paper")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print("Drive not mounted, writing to /content only:", exc)
PRED_CSV = OUT_DIR / "cv_predictions.csv"
FOLD_CSV = OUT_DIR / "cv_fold_metrics.csv"
print("Output dir:", OUT_DIR)

## 2. Pool every image and build the folds

The zip ships the original train/val/test directories. Cross-validation needs all 746
images in one pool, re-partitioned by group. `manifest.csv` already carries the class,
the label and the `group_id` (case id for pathology, video id for normals).

In [ ]:
DATA = Path("data")
manifest = pd.read_csv(DATA / "manifest.csv")
manifest["path"] = [str(DATA / s / c / f) for s, c, f in
                    zip(manifest["split"], manifest["class"], manifest["filename"])]
missing = [p for p in manifest["path"] if not Path(p).exists()]
assert not missing, f"{len(missing)} files from the manifest are missing, e.g. {missing[:3]}"

y = manifest["label"].to_numpy()            # 1 = endometriosis, 0 = normal
groups = manifest["group_id"].to_numpy()
print(f"{len(manifest)} images | {manifest['group_id'].nunique()} groups | "
      f"{int(y.sum())} endometriosis / {int((y == 0).sum())} normal")

outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for fold, (tr_val_idx, test_idx) in enumerate(outer.split(manifest["path"], y, groups)):
    # inner group-wise split of the remaining data -> validation set for early stopping
    inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    tr_rel, val_rel = next(inner.split(tr_val_idx, y[tr_val_idx], groups[tr_val_idx]))
    train_idx, val_idx = tr_val_idx[tr_rel], tr_val_idx[val_rel]

    # leakage guards: no group may be shared between any two parts
    g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
    assert not (g_tr & g_va), "group leaks train<->val"
    assert not (g_tr & g_te), "group leaks train<->test"
    assert not (g_va & g_te), "group leaks val<->test"

    FOLDS.append({"train": train_idx, "val": val_idx, "test": test_idx})
    print(f"fold {fold}: train {len(train_idx):3d} ({int(y[train_idx].sum())} endo) | "
          f"val {len(val_idx):3d} ({int(y[val_idx].sum())} endo) | "
          f"test {len(test_idx):3d} ({int(y[test_idx].sum())} endo) | "
          f"groups {len(g_tr)}/{len(g_va)}/{len(g_te)}")

all_test = np.concatenate([f["test"] for f in FOLDS])
assert len(set(all_test)) == len(manifest), "the folds must cover every image exactly once"
print("\nLeakage checks passed; the 5 test parts tile the whole dataset.")

## 3. Dataset, models and the training loop

Same recipe as the main experiment: ImageNet weights, full fine-tuning, Adam,
light augmentation on train only, early stopping on **validation AUC**.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class FrameDataset(Dataset):
    """Indexes the pooled manifest, so folds are plain index arrays."""

    def __init__(self, indices, train: bool):
        self.paths = manifest["path"].to_numpy()[indices]
        self.labels = y[indices]
        self.tf = train_tf if train else eval_tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), int(self.labels[i])


def build_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


def build_efficientnet_b0():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m


def build_vit_b16():
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    m.heads.head = nn.Linear(m.heads.head.in_features, 2)
    return m


MODELS = {                      # name -> (builder, learning rate)
    "ResNet50": (build_resnet50, 1e-4),
    "EfficientNet-B0": (build_efficientnet_b0, 1e-4),
    "ViT-B/16": (build_vit_b16, 2e-5),
}
POS = 1                         # column 1 of the softmax = endometriosis


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, trues = [], []
    for x, t in loader:
        p = torch.softmax(model(x.to(device)), 1)[:, POS].cpu().numpy()
        probs += p.tolist(); trues += t.numpy().tolist()
    return np.asarray(probs), np.asarray(trues)


def train_one(model, lr, train_loader, val_loader, tag):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best_auc, best_state, wait = 0.0, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        for x, t in train_loader:
            x, t = x.to(device), t.to(device)
            opt.zero_grad(); crit(model(x), t).backward(); opt.step()
        val_prob, val_true = predict(model, val_loader)
        auc = roc_auc_score(val_true, val_prob)
        if auc > best_auc:
            best_auc, wait = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"    {tag}: early stop @ epoch {ep + 1} (best val AUC {best_auc:.3f})")
                break
    model.load_state_dict(best_state)
    return model, best_auc

## 4. Run the 15 fits

Each `(model, fold)` result is appended to `cv_predictions.csv` as soon as it finishes.
Re-running this cell skips whatever is already in that file, so a Colab disconnect costs
at most one run.

In [ ]:
def done_runs():
    if not PRED_CSV.exists():
        return set()
    seen = pd.read_csv(PRED_CSV)
    return set(zip(seen["model"], seen["fold"]))


def append_rows(path, rows, header):
    exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if not exists:
            w.writerow(header)
        w.writerows(rows)


already = done_runs()
if already:
    print(f"Resuming: {len(already)} of {len(MODELS) * N_FOLDS} runs already finished")

for name, (builder, lr) in MODELS.items():
    for fold, split in enumerate(FOLDS):
        if (name, fold) in already:
            print(f"[{name} fold {fold}] skipped (already done)")
            continue
        t0 = time.time()
        # fresh seed per run so folds are reproducible independently of order
        torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)
        np.random.seed(SEED + fold); random.seed(SEED + fold)

        train_loader = DataLoader(FrameDataset(split["train"], True), BATCH,
                                  shuffle=True, num_workers=2, drop_last=False)
        val_loader = DataLoader(FrameDataset(split["val"], False), BATCH,
                                shuffle=False, num_workers=2)
        test_loader = DataLoader(FrameDataset(split["test"], False), BATCH,
                                 shuffle=False, num_workers=2)

        print(f"[{name} fold {fold}] training on {len(split['train'])} images...")
        model, val_auc = train_one(builder(), lr, train_loader, val_loader, f"{name} fold {fold}")

        prob, true = predict(model, test_loader)
        pred = (prob >= 0.5).astype(int)
        metrics = {
            "Accuracy": accuracy_score(true, pred),
            "Precision": precision_score(true, pred, zero_division=0),
            "Recall": recall_score(true, pred, zero_division=0),
            "F1": f1_score(true, pred, zero_division=0),
            "AUC": roc_auc_score(true, prob),
        }
        print(f"    done in {time.time() - t0:.0f}s | test AUC {metrics['AUC']:.3f} "
              f"acc {metrics['Accuracy']:.3f} recall {metrics['Recall']:.3f}")

        idx = split["test"]
        append_rows(PRED_CSV,
                    [[name, fold, manifest["filename"].to_numpy()[i],
                      manifest["group_id"].to_numpy()[i], int(y[i]), f"{p:.6f}", int(p >= 0.5)]
                     for i, p in zip(idx, prob)],
                    ["model", "fold", "filename", "group_id", "y_true", "prob", "pred"])
        append_rows(FOLD_CSV,
                    [[name, fold, len(split["train"]), len(split["test"]), round(val_auc, 4)]
                     + [round(metrics[k], 4) for k in
                        ("Accuracy", "Precision", "Recall", "F1", "AUC")]],
                    ["model", "fold", "n_train", "n_test", "val_AUC",
                     "Accuracy", "Precision", "Recall", "F1", "AUC"])
        del model
        torch.cuda.empty_cache()

print("\nAll runs complete.")

## 5. Aggregate

Two numbers matter per model:

- **mean ± std across the 5 folds** — the robustness claim for the paper
- **pooled out-of-fold AUC** — every one of the 746 images predicted by a model that
  never saw its group, which is a single, tighter estimate than any one fold

In [ ]:
folds = pd.read_csv(FOLD_CSV).drop_duplicates(subset=["model", "fold"], keep="last")
preds = pd.read_csv(PRED_CSV).drop_duplicates(subset=["model", "fold", "filename"], keep="last")

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
summary = folds.groupby("model")[metric_cols].agg(["mean", "std"]).round(4)
print("=== 5-fold cross-validation (mean +- std) ===")
for name in MODELS:
    if name not in summary.index:
        continue
    row = summary.loc[name]
    print(f"{name:16s} " + "  ".join(
        f"{m} {row[(m, 'mean')]:.3f}+-{row[(m, 'std')]:.3f}" for m in metric_cols))

print("\n=== Pooled out-of-fold (all 746 images) ===")
pooled_rows = []
for name in MODELS:
    sub = preds[preds["model"] == name]
    if sub.empty:
        continue
    auc = roc_auc_score(sub["y_true"], sub["prob"])
    acc = accuracy_score(sub["y_true"], sub["pred"])
    rec = recall_score(sub["y_true"], sub["pred"], zero_division=0)
    prec = precision_score(sub["y_true"], sub["pred"], zero_division=0)
    f1 = f1_score(sub["y_true"], sub["pred"], zero_division=0)
    pooled_rows.append([name, len(sub), round(acc, 4), round(prec, 4), round(rec, 4),
                        round(f1, 4), round(auc, 4)])
    print(f"{name:16s} n={len(sub)}  AUC {auc:.3f}  acc {acc:.3f}  recall {rec:.3f}")

summary.to_csv(OUT_DIR / "cv_summary.csv")
pd.DataFrame(pooled_rows, columns=["model", "n", "Accuracy", "Precision", "Recall", "F1", "AUC"]
             ).to_csv(OUT_DIR / "cv_pooled.csv", index=False)
print("\nSaved cv_summary.csv and cv_pooled.csv in", OUT_DIR)

## 6. Download the four CSVs

Put these in the repo's `results/` folder; the local analysis scripts take it from there.

In [ ]:
from google.colab import files
for f in ["cv_predictions.csv", "cv_fold_metrics.csv", "cv_summary.csv", "cv_pooled.csv"]:
    p = OUT_DIR / f
    if p.exists():
        files.download(str(p))
    else:
        print("missing:", p)